# Random Forest Churn Prediction
### Course template: Machine Learning for Business Insights (Module 5)

**What this notebook does**
1. Loads your customer CSV and defines a yes/no churn target
2. Builds a leakage-safe pipeline
3. Trains a baseline and a default random forest
4. Tunes the forest with **RandomizedSearchCV + cross-validation**
5. Evaluates with precision, recall, F1, ROC-AUC, and an **expected-value threshold table**
6. Explains the model with feature importance and partial dependence

**Why a random forest?** One decision tree is readable but jumpy: small data changes reshape it.
A forest trains hundreds of slightly different trees and **averages their votes**, like asking 300
experienced reps instead of one. It's usually more accurate and more stable, at the cost of
not having one simple rulebook.

Edit **only the CONFIG cell**, then run top to bottom.

In [ ]:
# ============================================================
# IMPORTS (scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    cross_validate, cross_val_predict, RandomizedSearchCV, learning_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.calibration import calibration_curve
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
RANDOM_STATE = 42
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: EDIT THIS CELL
# ============================================================
CSV_PATH = None                     # e.g. "/content/customers.csv"; None = demo

TARGET_COLUMN = "churned"
POSITIVE_VALUE = 1                  # value meaning "churned" (1, "Yes", "churned", ...)

NUMERIC_COLUMNS = ["tenure_months", "monthly_charge", "support_tickets_90d",
                   "logins_30d", "late_payments_12m", "discount_pct"]
CATEGORICAL_COLUMNS = ["plan", "contract_type", "region", "acquisition_channel"]

# IDs and LEAKAGE columns. Churn leakage is common, so watch for:
# "cancellation_reason", "final_bill", "account_status", "days_since_last_login"
# measured AFTER the prediction date, "retention_offer_given".
DROP_COLUMNS = ["customer_id"]

# Metric used to pick the best settings.
# "average_precision" (PR-AUC) is best when churn is rare (<20%); "roc_auc" otherwise.
MAIN_METRIC = "average_precision"

# --- Business numbers for the threshold table -------------------
VALUE_IF_SAVED = 600          # value of keeping one customer (e.g. 12 months of margin)
ACTION_COST = 40              # cost per customer contacted (call time, discount, gift)
ACTION_SUCCESS_RATE = 0.30    # share of contacted would-be churners you actually save
TEAM_CAPACITY = None          # max customers your team can contact per cycle (None = no cap)

DATA_SIZE_MODE = "auto"       # "auto", "small", "large"
SAMPLE_ROWS_FOR_SEARCH = None # large data: tune on a sample (e.g. 50_000), then refit on all
N_SEARCH_ITER = 30            # random combinations to try (raise to 60+ if time allows)
TEST_SIZE = 0.2
print("Config set.")

In [ ]:
# ============================================================
# LOAD DATA. Demo data is SYNTHETIC.
# ============================================================
def make_demo(n=2_000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    tenure = rng.integers(1, 72, n)
    plan = rng.choice(["basic", "standard", "premium"], n, p=[0.4, 0.4, 0.2])
    contract = rng.choice(["monthly", "annual", "two_year"], n, p=[0.55, 0.3, 0.15])
    charge = (np.select([plan == "basic", plan == "standard"], [30, 55], 90) + rng.normal(0, 6, n)).round(2)
    tickets = rng.poisson(1.0, n)
    logins = rng.poisson(12, n)
    late = rng.poisson(0.4, n)
    disc = rng.choice([0, 0, 0, 10, 20], n)
    region = rng.choice(["north", "south", "east", "west"], n)
    channel = rng.choice(["web", "referral", "partner", "paid_ads"], n)
    # Non-linear patterns + interactions: what forests are good at
    logit = (-1.6 + 1.3 * (contract == "monthly") - 0.035 * tenure
             + 0.45 * np.minimum(tickets, 4) - 0.08 * np.minimum(logins, 20)
             + 0.9 * ((tickets >= 3) & (logins < 6))      # frustrated AND disengaged
             + 0.5 * late + 0.012 * (charge - 55) + 0.4 * (channel == "paid_ads"))
    churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame({"customer_id": range(10_000, 10_000 + n), "tenure_months": tenure,
                       "monthly_charge": charge, "support_tickets_90d": tickets, "logins_30d": logins,
                       "late_payments_12m": late, "discount_pct": disc, "plan": plan,
                       "contract_type": contract, "region": region, "acquisition_channel": channel,
                       "churned": churn})
    df.loc[rng.choice(n, 40, replace=False), "logins_30d"] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo(); print("Using DEMO data (synthetic).")
else:
    df = pd.read_csv(CSV_PATH); print(f"Loaded {CSV_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK + TARGET (_y = 1 churned, 0 stayed)
# ============================================================
need = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
miss = [c for c in need if c not in df.columns]
if miss:
    raise ValueError(f"Not found: {miss}\nYour columns: {list(df.columns)}")
overlap = set(DROP_COLUMNS) & set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS)
if overlap:
    raise ValueError(f"These are in DROP_COLUMNS and also listed as inputs: {overlap}")

for c in NUMERIC_COLUMNS:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[$,%\s]", "", regex=True), errors="coerce")
        print(f"Converted '{c}' to numbers.")
# Store numeric inputs as decimals (floats); partial dependence charts refuse whole-number columns
df[NUMERIC_COLUMNS] = df[NUMERIC_COLUMNS].astype(float)

df = df.dropna(subset=[TARGET_COLUMN])
print("Target values found:", df[TARGET_COLUMN].value_counts().to_dict())
df["_y"] = (df[TARGET_COLUMN].astype(str).str.strip().str.lower()
            == str(POSITIVE_VALUE).strip().lower()).astype(int)
if df["_y"].nunique() < 2:
    raise ValueError("Only one class after mapping. Check POSITIVE_VALUE.")
print(f"Churn rate: {df['_y'].mean():.1%} ({df['_y'].sum():,} churners)")

## Section 1: Quick EDA
We're checking three things: data quality, how rare churn is, and whether any input looks
**suspiciously perfect** (a classic sign of leakage).

In [ ]:
# ============================================================
# DATA QUALITY
# ============================================================
m = df[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS].isna().sum()
print("Missing values:\n", m[m > 0] if m.sum() else "None")
print("Duplicate rows:", df.duplicated().sum())
display(df[NUMERIC_COLUMNS].describe().T)

# ============================================================
# CHURN RATE BY CATEGORY (quick business story)
# ============================================================
fig, axes = plt.subplots(1, len(CATEGORICAL_COLUMNS), figsize=(4 * len(CATEGORICAL_COLUMNS), 3.2))
for ax, c in zip(np.atleast_1d(axes), CATEGORICAL_COLUMNS):
    df.groupby(c)["_y"].mean().sort_values().plot.barh(ax=ax, color="indianred")
    ax.set_title(f"Churn rate by {c}"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

# ============================================================
# LEAKAGE SNIFF TEST: single-input ROC-AUC
# One input alone scoring > 0.90 is a red flag: is it known BEFORE churn?
# ============================================================
print("\nSingle-input ROC-AUC (how well each input ALONE separates churners):")
for c in NUMERIC_COLUMNS:
    s = df[[c, "_y"]].dropna()
    auc = roc_auc_score(s["_y"], s[c]); auc = max(auc, 1 - auc)
    flag = "  ⚠ suspiciously strong: check for leakage!" if auc > 0.90 else ""
    print(f"  {c:<25} {auc:.2f}{flag}")

In [ ]:
# ============================================================
# DATA SIZE CHECK: what matters is the number of CHURNERS
# ============================================================
n, n_pos = len(df), int(df["_y"].sum())
MODE = ("small" if n_pos < 200 else "large") if DATA_SIZE_MODE == "auto" else DATA_SIZE_MODE
print(f"Rows {n:,} | Churners {n_pos:,} | Mode: {MODE.upper()}")
if MODE == "small":
    print("""SMALL-DATA PATH
- Repeated stratified CV (results swing a lot with few churners)
- Smaller search, stronger tree constraints (shallower trees, bigger leaves)
- Compare against simple logistic regression; forests don't always win on small data""")
if n > 200_000 and SAMPLE_ROWS_FOR_SEARCH is None:
    print("⚠ Large dataset: consider SAMPLE_ROWS_FOR_SEARCH = 50_000 to speed up tuning.")

In [ ]:
# ============================================================
# SPLIT + CV. Stratified so every fold has the same churn rate.
# TIME NOTE: if your data spans many months, a time-based split
# (train on older customers, test on the most recent) is more honest.
# ============================================================
feature_cols = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X, y = df[feature_cols], df["_y"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

CV = (RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE) if MODE == "small"
      else StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
print(f"Train {len(X_train):,} ({y_train.mean():.1%} churn) | Test {len(X_test):,} ({y_test.mean():.1%} churn)")

## Section 2: Leakage-Safe Pipeline
Forests, like single trees, **don't need scaling**, and ordinal-encoded categories work well.
All filling and encoding happens **inside** the pipeline, so cross-validation never peeks at validation data.

In [ ]:
# ============================================================
# PREPROCESSOR
# ============================================================
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC_COLUMNS),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), CATEGORICAL_COLUMNS),
], verbose_feature_names_out=False)

def make_rf(**params):
    return Pipeline([("prep", preprocessor),
                     ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params))])

# A simple logistic regression for comparison (needs scaling + one-hot)
logit_prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUMERIC_COLUMNS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("oh", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))]),
     CATEGORICAL_COLUMNS),
])
logit_model = Pipeline([("prep", logit_prep), ("model", LogisticRegression(max_iter=2000))])
print("Pipelines ready.")

## Section 3: Baselines
- **Dummy:** always predicts "no churn". Every real model must beat it.
- **Logistic regression:** the simple, explainable benchmark from Module 4.
- **Default random forest:** 100 unlimited trees. Usually decent, but trees can memorize individual customers.

In [ ]:
# ============================================================
# BASELINE COMPARISON
# ============================================================
SCORING = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision",
           "F1": "f1", "Precision": "precision", "Recall": "recall"}

def cv_summary(model, name, Xd=X_train, yd=y_train):
    r = cross_validate(model, Xd, yd, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=1)
    row = {"model": name}
    for k in SCORING:
        row[k] = r[f"test_{k}"].mean(); row[f"{k} ±"] = r[f"test_{k}"].std()
    row["ROC-AUC (train)"] = r["train_ROC-AUC"].mean()
    return row

res = pd.DataFrame([
    cv_summary(Pipeline([("prep", preprocessor), ("model", DummyClassifier(strategy="prior"))]), "Baseline (dummy)"),
    cv_summary(logit_model, "Logistic regression"),
    cv_summary(make_rf(), "Default random forest"),
]).set_index("model")
display(res)
print(f"""
HOW TO READ THIS
- PR-AUC for a random guess = churn rate ({y_train.mean():.2f}). Above that = real skill.
- ROC-AUC 0.5 = coin flip; 0.7–0.8 = useful; 0.8+ = strong (or check for leakage).
- F1/Precision/Recall here use a 0.5 cutoff, which is often wrong for business. We fix that in Section 6.
- "ROC-AUC (train)" near 1.00 for the forest is normal (trees memorize training data);
  judge it on the validation columns only.
""")

## Section 4: Tuning with RandomizedSearchCV

Instead of trying every combination (slow), **random search** tries a set number of random combos.
It usually finds near-best settings in a fraction of the time.

| Setting | Plain meaning | Overfitting control |
|---|---|---|
| `n_estimators` | Number of trees | More = more stable, never more overfit; just slower |
| `max_depth` | How deep each tree goes | Shallower = less memorizing |
| `min_samples_leaf` | Minimum customers per final group | Bigger = smoother, safer |
| `max_features` | Share of inputs each split may consider | Lower = more diverse trees |
| `max_samples` | Share of customers each tree sees | Lower = more diverse, faster |
| `class_weight` | Extra weight on churners | Helps when churn is rare |

⚠️ **Small data:** a forest can't fix too few churners. Keep leaves big (≥ 10–20 customers), cap depth (≤ 6–8),
and take the search winner's margin with a grain of salt. If it doesn't clearly beat logistic regression,
use logistic regression (simpler, explainable).

In [ ]:
# ============================================================
# SEARCH SPACE
# randint(a, b) = random whole number from a to b-1
# ============================================================
if MODE == "small":
    space = {
        "model__n_estimators": [200, 300],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__min_samples_leaf": randint(10, 40),
        "model__max_features": ["sqrt", 0.3, 0.5],
        "model__class_weight": [None, "balanced", "balanced_subsample"],
    }
    n_iter = min(N_SEARCH_ITER, 25)
else:
    space = {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [None, 6, 8, 12, 16, 24],
        "model__min_samples_leaf": randint(1, 50),
        "model__min_samples_split": randint(2, 40),
        "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
        "model__max_samples": [None, 0.5, 0.7],
        "model__class_weight": [None, "balanced", "balanced_subsample"],
    }
    n_iter = N_SEARCH_ITER

# Large data: tune on a stratified sample to save time
if SAMPLE_ROWS_FOR_SEARCH and len(X_train) > SAMPLE_ROWS_FOR_SEARCH:
    Xs, _, ys, _ = train_test_split(X_train, y_train, train_size=SAMPLE_ROWS_FOR_SEARCH,
                                    stratify=y_train, random_state=RANDOM_STATE)
    print(f"Tuning on a {len(Xs):,}-row sample.")
else:
    Xs, ys = X_train, y_train

search = RandomizedSearchCV(make_rf(), space, n_iter=n_iter, cv=CV, scoring=MAIN_METRIC,
                            random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True)
search.fit(Xs, ys)

tbl = pd.DataFrame(search.cv_results_)
pcols = [c for c in tbl.columns if c.startswith("param_")]
tbl = tbl[pcols + ["mean_test_score", "std_test_score", "mean_train_score"]]
tbl.columns = [c.replace("param_model__", "") for c in pcols] + ["valid", "±", "train"]
tbl["gap (train − valid)"] = tbl["train"] - tbl["valid"]
# kind="stable" keeps ties in their original order, so row positions line up with cv_results_ below
tbl = tbl.sort_values("valid", ascending=False, kind="stable").reset_index(drop=True)
print(f"Top 10 of {n_iter} combinations ({MAIN_METRIC}):"); display(tbl.head(10))

# ============================================================
# SIMPLICITY RULE: among combos within one spread of the best,
# prefer the one with the SMALLEST train-validation gap (least overfit).
# ============================================================
best = tbl.iloc[0]
within = tbl[tbl["valid"] >= best["valid"] - best["±"]]
chosen_idx = within["gap (train − valid)"].idxmin()
order = np.argsort(-search.cv_results_["mean_test_score"], kind="stable")
BEST_PARAMS = {k.replace("model__", ""): v
               for k, v in search.cv_results_["params"][order[chosen_idx]].items()}
print(f"\nBest score: {best['valid']:.3f} ± {best['±']:.3f}")
print(f"CHOSEN (within one spread, least overfit): {BEST_PARAMS}")
print(f"  valid {within.loc[chosen_idx, 'valid']:.3f}, gap {within.loc[chosen_idx, 'gap (train − valid)']:.3f}")

In [ ]:
# ============================================================
# COMPARE: tuned forest vs. baselines (full training set)
# ============================================================
final_model = make_rf(**BEST_PARAMS)
res2 = pd.concat([res, pd.DataFrame([cv_summary(final_model, "Tuned random forest")]).set_index("model")])
display(res2[["ROC-AUC", "ROC-AUC ±", "PR-AUC", "PR-AUC ±", "ROC-AUC (train)"]])

lr, rf = res2.loc["Logistic regression", "PR-AUC"], res2.loc["Tuned random forest", "PR-AUC"]
spread = res2.loc["Tuned random forest", "PR-AUC ±"]
if rf - lr < spread:
    print("⚖ The forest is NOT clearly better than logistic regression. Consider the simpler model.")
else:
    print(f"✅ The forest beats logistic regression by {rf - lr:.3f} PR-AUC, more than the spread.")

## Section 5: Final Test-Set Evaluation
Fit on all training data, score the locked test set **once**.

In [ ]:
# ============================================================
# TEST SET: fit on all training rows, score the locked test set ONCE
# ============================================================
final_model.fit(X_train, y_train)
proba_test = final_model.predict_proba(X_test)[:, 1]

print(f"""=== TEST SET (ranking quality) ===
ROC-AUC: {roc_auc_score(y_test, proba_test):.3f}
PR-AUC:  {average_precision_score(y_test, proba_test):.3f}  (random = {y_test.mean():.3f})
""")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, proba_test, ax=ax[0], name="Random forest")
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=ax[1], name="Random forest")
ax[1].axhline(y_test.mean(), color="k", ls="--", lw=0.8, label="random"); ax[1].legend()
ax[1].set_title("Precision–recall curve")
plt.tight_layout(); plt.show()

# Compare with the logistic benchmark on the same test set (for the record)
logit_model.fit(X_train, y_train)
p_lr = logit_model.predict_proba(X_test)[:, 1]
print(f"Logistic benchmark on test: ROC-AUC {roc_auc_score(y_test, p_lr):.3f} | PR-AUC {average_precision_score(y_test, p_lr):.3f}")

## Section 6: Who Should We Call? (Threshold, Capacity, and Expected Value)

The forest gives each customer a churn probability. The **threshold** decides who gets a retention action.

**Expected value = churners reached × VALUE_IF_SAVED × ACTION_SUCCESS_RATE − customers contacted × ACTION_COST**

- We choose the threshold on **out-of-fold** predictions from the training data (each customer scored by
  a model that never saw them), then check it **once** on the test set.
- If you set `TEAM_CAPACITY`, thresholds that would flag more customers than your team can handle are ruled out.
- Numbers are scaled to your **full customer file** so they read as "per campaign cycle".

In [ ]:
# ============================================================
# OUT-OF-FOLD PROBABILITIES + THRESHOLD PROFIT TABLE
# ============================================================
OOF_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)   # each row predicted once
proba_oof = cross_val_predict(final_model, X_train, y_train, cv=OOF_CV, method="predict_proba")[:, 1]

CUSTOMER_BASE = len(df)            # customers scored per cycle (assumed = rows in your file)
BREAK_EVEN = ACTION_COST / (VALUE_IF_SAVED * ACTION_SUCCESS_RATE)

def threshold_table(y_true, proba, value=VALUE_IF_SAVED, cost=ACTION_COST, success=ACTION_SUCCESS_RATE):
    """Expected value of contacting everyone at or above each threshold (scaled to CUSTOMER_BASE)."""
    y_true = np.asarray(y_true)
    cands = list(np.round(np.arange(0.05, 0.951, 0.025), 3))
    if TEAM_CAPACITY:
        cands.append(float(np.quantile(proba, 1 - min(TEAM_CAPACITY / CUSTOMER_BASE, 1.0))))
    scale = CUSTOMER_BASE / len(y_true)
    rows = []
    for t in sorted(set(cands)):
        flag = proba >= t
        nf, tp = int(flag.sum()), int((flag & (y_true == 1)).sum())
        rows.append({"threshold": t, "contacted": round(nf * scale), "churners reached": round(tp * scale),
                     "precision": tp / nf if nf else np.nan, "recall": tp / max(y_true.sum(), 1),
                     "expected value": (tp * value * success - nf * cost) * scale,
                     "within capacity": (TEAM_CAPACITY is None) or (nf * scale <= TEAM_CAPACITY * 1.001)})
    return pd.DataFrame(rows)

tt = threshold_table(y_train, proba_oof)
ok = tt[tt["within capacity"]]
best_row = ok.loc[ok["expected value"].idxmax()]
THRESHOLD = float(best_row["threshold"])

display(tt.iloc[::2].style.format({"threshold": "{:.3f}", "precision": "{:.2f}", "recall": "{:.2f}",
                                    "contacted": "{:,.0f}", "churners reached": "{:,.0f}",
                                    "expected value": "${:,.0f}"}).hide(axis="index"))
print(f"""
Break-even precision: {BREAK_EVEN:.1%} (a contact pays off only if at least this share of contacted customers would churn)
CHOSEN THRESHOLD: {THRESHOLD:.3f}{'  (capacity-limited)' if TEAM_CAPACITY else ''}
→ Contact ~{best_row['contacted']:,.0f} of {CUSTOMER_BASE:,} customers; reach {best_row['recall']:.0%} of churners at {best_row['precision']:.0%} precision.
→ Estimated value per cycle: ${best_row['expected value']:,.0f}
""")

plt.figure(figsize=(8, 4))
plt.plot(tt["threshold"], tt["expected value"], "o-")
plt.axvline(THRESHOLD, color="red", ls="--", label=f"chosen {THRESHOLD:.2f}")
plt.axhline(0, color="k", lw=0.8); plt.xlabel("Threshold"); plt.ylabel("Expected value per cycle ($)")
plt.title("Retention program value by threshold (out-of-fold)"); plt.legend(); plt.show()

# One honest check on the test set
tt_test = threshold_table(y_test, proba_test)
r = tt_test.iloc[(tt_test["threshold"] - THRESHOLD).abs().argmin()]
print(f"TEST SET at {THRESHOLD:.3f}: precision {r['precision']:.0%}, recall {r['recall']:.0%}, "
      f"estimated value ${r['expected value']:,.0f} per cycle")

In [ ]:
# ============================================================
# SENSITIVITY: is the decision robust to our business estimates?
# ============================================================
sens = []
for v in [0.5, 1.0, 1.5]:
    for s in [0.5, 1.0, 1.5]:
        t_ = threshold_table(y_train, proba_oof, value=VALUE_IF_SAVED * v, success=min(ACTION_SUCCESS_RATE * s, 1))
        t_ = t_[t_["within capacity"]]; b = t_.loc[t_["expected value"].idxmax()]
        sens.append({"value if saved": VALUE_IF_SAVED * v, "success rate": min(ACTION_SUCCESS_RATE * s, 1),
                     "best threshold": b["threshold"], "contacted": b["contacted"], "expected value": b["expected value"]})
display(pd.DataFrame(sens).style.format({"value if saved": "${:,.0f}", "success rate": "{:.0%}", "best threshold": "{:.3f}",
                                          "contacted": "{:,.0f}", "expected value": "${:,.0f}"}).hide(axis="index"))

# ============================================================
# LIFT BY DECILE (test set): decile 1 = top 10% riskiest customers
# ============================================================
lift = pd.DataFrame({"y": y_test.values, "p": proba_test})
lift["decile"] = pd.qcut(lift["p"].rank(method="first", ascending=False), 10, labels=range(1, 11))
lt = lift.groupby("decile", observed=True).agg(customers=("y", "size"), churners=("y", "sum"), avg_score=("p", "mean"))
lt["churn rate"] = lt["churners"] / lt["customers"]
lt["lift"] = lt["churn rate"] / lift["y"].mean()
lt["cumulative % of churners"] = lt["churners"].cumsum() / lt["churners"].sum() * 100
display(lt.style.format({"avg_score": "{:.2f}", "churn rate": "{:.1%}", "lift": "{:.1f}x", "cumulative % of churners": "{:.0f}%"}))
print(f"Calling the top 20% reaches {lt['cumulative % of churners'].iloc[1]:.0f}% of churners "
      f"(random calling would reach 20%).")

# ============================================================
# CALIBRATION: do the probabilities mean what they say?
# Forests tend to squeeze probabilities toward the middle, and
# class_weight="balanced" pushes them up. Ranking is still fine; if you
# need true probabilities (e.g. expected-loss math), use
# sklearn.calibration.CalibratedClassifierCV.
# ============================================================
fp, mp = calibration_curve(y_test, proba_test, n_bins=8, strategy="quantile")
plt.figure(figsize=(5, 5)); plt.plot(mp, fp, "o-", label="forest"); plt.plot([0, 1], [0, 1], "k--", label="perfect")
plt.xlabel("Predicted churn probability"); plt.ylabel("Actual churn rate"); plt.title("Calibration (test)"); plt.legend(); plt.show()

## Section 7: Why Do Customers Churn? (Interpretation)

A forest has no single rulebook, so we use three lenses:

1. **Permutation importance** (preferred): shuffle one input on the test set and see how much the score drops.
2. **Built-in importance**: quick, but biased toward inputs with many distinct values (like charges or tenure).
3. **Partial dependence (PDP) + ICE**: *how* churn risk changes as one input moves, holding the rest of each
   customer's profile as-is. The thick line is the average; thin lines are individual customers.

⚠️ These show what the model learned from history, not guaranteed cause and effect.

In [ ]:
# ============================================================
# PERMUTATION vs. BUILT-IN IMPORTANCE
# ============================================================
perm = permutation_importance(final_model, X_test, y_test, scoring=MAIN_METRIC,
                              n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1)
rf = final_model.named_steps["model"]
feat = list(final_model.named_steps["prep"].get_feature_names_out())
imp = pd.DataFrame({"permutation (score drop)": perm.importances_mean, "±": perm.importances_std},
                   index=feature_cols)
imp["built-in"] = pd.Series(rf.feature_importances_, index=feat).reindex(feature_cols).values
imp = imp.sort_values("permutation (score drop)", ascending=False)
display(imp)

fig, ax = plt.subplots(1, 2, figsize=(12, 0.45 * len(imp) + 2))
imp["permutation (score drop)"][::-1].plot.barh(ax=ax[0], xerr=imp["±"][::-1], color="steelblue")
ax[0].set_title(f"Permutation importance (drop in {MAIN_METRIC})")
imp["built-in"][::-1].plot.barh(ax=ax[1], color="gray"); ax[1].set_title("Built-in importance")
plt.tight_layout(); plt.show()
weak = imp.index[imp["permutation (score drop)"] <= imp["±"]].tolist()
print("Inputs whose importance is within noise (could be dropped to simplify):", weak or "none")

In [ ]:
# ============================================================
# PARTIAL DEPENDENCE + ICE for the top numeric drivers
# Read it like: "as support tickets go from 0 to 4, predicted churn
# risk rises from about X to Y for a typical customer."
# ============================================================
from sklearn.inspection import partial_dependence

top_num = [c for c in imp.index if c in NUMERIC_COLUMNS][:3]
fig, ax = plt.subplots(1, len(top_num), figsize=(5 * len(top_num), 4))
PartialDependenceDisplay.from_estimator(
    final_model, X_test, features=top_num, kind="both", subsample=60,
    random_state=RANDOM_STATE, ax=np.atleast_1d(ax), grid_resolution=20,
    ice_lines_kw={"alpha": 0.15}, pd_line_kw={"color": "red", "lw": 3})
plt.suptitle("How churn risk changes with each driver (red = average; grey = individual customers)")
plt.tight_layout(); plt.show()

# Plain-English direction for each top driver (from the average PDP line)
for c in top_num:
    pdp = partial_dependence(final_model, X_test, [c], grid_resolution=20)
    vals, grid = pdp["average"][0], pdp["grid_values"][0]
    keep = ~np.isnan(grid)                       # blanks can appear as an extra grid point; skip them
    vals, grid = vals[keep], grid[keep]
    trend = "rises" if vals[-1] > vals[0] else "falls"
    print(f"- As {c} goes from {grid[0]:,.0f} to {grid[-1]:,.0f}, average predicted churn {trend} "
          f"from {vals[0]:.0%} to {vals[-1]:.0%}.")

In [ ]:
# ============================================================
# IMPORTANCE STABILITY: do the top drivers stay on top?
# Refit the forest on several bootstrap samples and re-rank inputs.
# Stable top-3 = safe to talk about. Shuffling ranks = say "among the top drivers".
# ============================================================
N_REFITS = 5
ranks = []
rng = np.random.default_rng(RANDOM_STATE)
for b in range(N_REFITS):
    idx = rng.integers(0, len(X_train), len(X_train))
    m = clone(final_model).fit(X_train.iloc[idx], y_train.iloc[idx])
    pi = permutation_importance(m, X_test, y_test, scoring=MAIN_METRIC, n_repeats=5,
                                random_state=RANDOM_STATE, n_jobs=-1)
    ranks.append(pd.Series(pi.importances_mean, index=feature_cols).rank(ascending=False))
ranks = pd.concat(ranks, axis=1)
stab = pd.DataFrame({"median rank": ranks.median(1), "best rank": ranks.min(1), "worst rank": ranks.max(1)}).sort_values("median rank")
display(stab.astype(int))

## Optional: XGBoost or LightGBM

Gradient-boosting libraries often squeeze out a bit more accuracy on large tables. The steps are identical:
swap the model inside the pipeline and reuse every cell above.

```python
# !pip install lightgbm xgboost     # both are preinstalled in Colab
# from lightgbm import LGBMClassifier
# lgbm = Pipeline([("prep", preprocessor),
#                  ("model", LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
#                                           min_child_samples=30, subsample=0.8, subsample_freq=1,
#                                           colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1))])
# display(pd.DataFrame([cv_summary(lgbm, "LightGBM")]).set_index("model"))
#
# from xgboost import XGBClassifier
# xgb = Pipeline([("prep", preprocessor),
#                 ("model", XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4,
#                                         subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
#                                         random_state=RANDOM_STATE))])
```

Rule of thumb: switch only if the gain is **bigger than the ± spread**, and remember the boosting notebook
(Module 5) covers early stopping and tuning for these models.

## Section 8: Small Data vs. Large Data

**Small data (under ~200 churners):**
- Repeated stratified CV is used automatically; read every comparison against the ± spread
- Keep leaves big (`min_samples_leaf` ≥ 10–20) and depth modest; the search space already does this
- If the forest doesn't clearly beat logistic regression, ship the logistic model (simpler, explainable)
- Treat importance ranks beyond the top 2–3 as noise

**Large data (thousands of churners):**
- Use `SAMPLE_ROWS_FOR_SEARCH` (e.g. 50,000) to tune quickly, then refit on everything
- `max_samples=0.5` trains each tree on half the rows: faster with little accuracy loss
- More trees (300–500) only make results steadier, never more overfit, just slower
- Re-check calibration if you use probabilities directly, and retrain on a schedule (e.g. quarterly)

In [ ]:
# ============================================================
# SAVE THE MODEL + CALL LIST
# ============================================================
import joblib

# Call list. DEMO NOTE: we score every row in the file. In real use, score your
# CURRENT active customers (same columns), not the historical training file.
ID_COLUMN = next((c for c in DROP_COLUMNS if c in df.columns), None)
calls = df[[ID_COLUMN] if ID_COLUMN else []].copy()
calls["churn_probability"] = final_model.predict_proba(df[feature_cols])[:, 1]
calls["expected_value_of_call"] = calls["churn_probability"] * VALUE_IF_SAVED * ACTION_SUCCESS_RATE - ACTION_COST
calls = calls[calls["churn_probability"] >= THRESHOLD].sort_values("churn_probability", ascending=False)
if TEAM_CAPACITY:
    calls = calls.head(TEAM_CAPACITY)
calls.round(3).to_csv("churn_call_list.csv", index=False)
print(f"Saved: churn_call_list.csv ({len(calls):,} customers, highest risk first)")
display(calls.head(10))

# Model + threshold, refit on all labeled data for use going forward
final_model.fit(pd.concat([X_train, X_test]), pd.concat([y_train, y_test]))
joblib.dump({"model": final_model, "threshold": THRESHOLD, "features": feature_cols}, "churn_rf_model.joblib")
print("Saved: churn_rf_model.joblib")
# Later: b = joblib.load("churn_rf_model.joblib"); p = b["model"].predict_proba(new_df[b["features"]])[:, 1]

## Final Checklist

- [ ] Churn definition and scoring date are clear; no input is measured after the scoring date
- [ ] Leakage sniff test passed (no single input with ROC-AUC > 0.90 unexplained)
- [ ] Forest beats both the dummy baseline and logistic regression by more than the spread (or I chose logistic)
- [ ] Tuning used cross-validation; chosen setting is the least-overfit within one spread of the best
- [ ] Test set used once; ROC-AUC / PR-AUC reported
- [ ] Threshold chosen from out-of-fold expected value, within team capacity; sensitivity checked
- [ ] Top drivers confirmed by permutation importance and PDP, and they make business sense
- [ ] Call list delivered with a **control group** held out so we can measure real saves
- [ ] Model, threshold, and feature list saved with the training date